# 1. Business Understanding
---
El objetivo del proyecto es clasificar automáticamente los tipos de ataques y sus categorías utilizando descripciones técnicas y vulnerabilidades.

**Requisitos Funcionales:** Clasificación, identificación de categorías, análisis de vulnerabilidades, visualización, filtrado y predicción.

In [1]:
import sys

!{sys.executable} -m pip install pandas numpy matplotlib seaborn wordcloud scikit-learn imbalanced-learn tensorflow shap joblib deep-translator flask scipy --quiet

print("✅ Todas las dependencias instaladas correctamente")

You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
✅ Todas las dependencias instaladas correctamente


---
## 1. Importación de Librerías

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

# Feature Engineering
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

# Modelos ML
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

# Modelos no supervisados
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import silhouette_score

# Evaluación
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (classification_report, accuracy_score,
                             f1_score, recall_score, precision_score,
                             confusion_matrix, roc_auc_score)

# Balanceo
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTETomek

# Deep Learning
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Dense, Dropout, BatchNormalization,
                                      Conv1D, GlobalMaxPooling1D, LSTM,
                                      Reshape, Embedding)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

# Explicabilidad
import shap

# Persistencia
import joblib

print('✅ Librerías importadas correctamente')
print(f'TensorFlow version: {tf.__version__}')

✅ Librerías importadas correctamente
TensorFlow version: 2.20.0


---
## 3. Data Understanding (EDA)

In [10]:
df = pd.read_csv('Attack_Dataset.csv', sep=',')
print(f'Shape original: {df.shape}')
df.info()

FileNotFoundError: [Errno 2] No such file or directory: 'Attack_Dataset.csv'

In [8]:
# Estadísticas descriptivas generales
print('=== Valores nulos por columna ===')
print(df.isnull().sum())
print(f'\nTotal registros: {len(df)}')
print(f'Total columnas: {len(df.columns)}')

=== Valores nulos por columna ===


NameError: name 'df' is not defined

In [ ]:
# 2.1 Distribución de Categorías (Top 15)
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

top_categories = df['Category'].value_counts().head(15)
sns.barplot(x=top_categories.values, y=top_categories.index,
            hue=top_categories.index, palette='viridis', legend=False, ax=axes[0])
axes[0].set_title('Top 15 Categorías de Ataque', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Frecuencia')

# Distribución completa (boxplot de frecuencias)
freq = df['Category'].value_counts()
axes[1].hist(freq.values, bins=30, color='steelblue', edgecolor='white')
axes[1].set_title('Distribución de Frecuencias por Categoría', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Cantidad de registros por clase')
axes[1].set_ylabel('Cantidad de clases')
axes[1].axvline(freq.mean(), color='red', linestyle='--', label=f'Media: {freq.mean():.0f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('plots/distribucion_categorias.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Total de categorías: {df["Category"].nunique()}')
print(f'Clase más frecuente: {freq.idxmax()} ({freq.max()} registros)')
print(f'Clase menos frecuente: {freq.idxmin()} ({freq.min()} registros)')

In [ ]:
# 2.2 Distribución de longitud de descripciones
df['desc_length'] = df['Scenario Description'].fillna('').apply(len)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['desc_length'], bins=50, kde=True, color='indigo', ax=axes[0])
axes[0].set_title('Distribución de Longitud de Descripciones', fontweight='bold')
axes[0].set_xlabel('Caracteres')
axes[0].axvline(df['desc_length'].mean(), color='red', linestyle='--',
                label=f'Media: {df["desc_length"].mean():.0f}')
axes[0].legend()

# Longitud por categoría (Top 10)
top10_cat = df['Category'].value_counts().head(10).index
df_top10 = df[df['Category'].isin(top10_cat)]
sns.boxplot(data=df_top10, x='desc_length', y='Category', palette='Set2', ax=axes[1])
axes[1].set_title('Longitud de Descripción por Categoría (Top 10)', fontweight='bold')
axes[1].set_xlabel('Caracteres')

plt.tight_layout()
plt.savefig('plots/longitud_descripciones.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Min: {df["desc_length"].min()} | Max: {df["desc_length"].max()} | Media: {df["desc_length"].mean():.1f}')

In [ ]:
import os
os.makedirs('plots', exist_ok=True)
os.makedirs('model', exist_ok=True)

all_text = ' '.join(df['Scenario Description'].fillna('').tolist())
wc = WordCloud(width=900, height=400, background_color='white',
               colormap='viridis', max_words=100).generate(all_text)

plt.figure(figsize=(14, 5))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('WordCloud — Términos más frecuentes en Scenario Description', fontweight='bold')
plt.savefig('plots/wordcloud.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 2.4 Matriz de Co-ocurrencia: Category vs Target Type
top_cat_h = df['Category'].value_counts().head(10).index
top_tgt_h = df['Target Type'].value_counts().head(10).index
filtered_heat = df[df['Category'].isin(top_cat_h) & df['Target Type'].isin(top_tgt_h)]

co_matrix = pd.crosstab(filtered_heat['Category'], filtered_heat['Target Type'])

plt.figure(figsize=(14, 7))
sns.heatmap(co_matrix, cmap='Blues', annot=True, fmt='d', linewidths=0.5)
plt.title('Matriz de Co-ocurrencia: Categoría vs Target Type (Top 10)', fontweight='bold')
plt.tight_layout()
plt.savefig('plots/matriz_coocurrencia.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 2.5 Top MITRE Techniques
top_mitre = df['MITRE Technique'].fillna('N/A').value_counts().head(15)

plt.figure(figsize=(12, 5))
sns.barplot(x=top_mitre.values, y=top_mitre.index, palette='rocket')
plt.title('Top 15 Técnicas MITRE ATT&CK', fontweight='bold')
plt.xlabel('Frecuencia')
plt.tight_layout()
plt.savefig('plots/top_mitre.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Data Preparation

In [ ]:
# 4.1 Limpieza
df.drop(columns=['Unnamed: 15'], inplace=True, errors='ignore')
df.columns = df.columns.str.strip()

# Imputación de nulos en columnas de texto
text_cols = ['Title', 'Scenario Description', 'Vulnerability', 'MITRE Technique']
df[text_cols] = df[text_cols].fillna('')

# 4.2 Ingeniería de features: combinar columnas de texto
df['text'] = (
    df['Title'] + ' ' +
    df['Scenario Description'] + ' ' +
    df['Vulnerability'] + ' ' +
    df['MITRE Technique']
).str.lower()

# 4.3 Filtrar clases con al menos 10 instancias para mayor robustez
counts = df['Category'].value_counts()
valid_classes = counts[counts >= 10].index
df = df[df['Category'].isin(valid_classes)].copy()

print(f'Registros tras limpieza: {df.shape[0]}')
print(f'Clases válidas (>= 10 instancias): {df["Category"].nunique()}')

# Estadísticas de desbalanceo
freq_clean = df['Category'].value_counts()
print(f'\nRatio desbalanceo (max/min): {freq_clean.max() / freq_clean.min():.1f}x')
print(f'Media por clase: {freq_clean.mean():.1f}')
print(f'Desviación estándar: {freq_clean.std():.1f}')

In [ ]:
# 4.4 Visualizar desbalanceo de clases DESPUÉS de limpieza
plt.figure(figsize=(14, 6))
freq_clean.plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Distribución de Clases — Dataset Limpio', fontweight='bold')
plt.xlabel('Categoría')
plt.ylabel('Frecuencia')
plt.xticks(rotation=90, fontsize=7)
plt.axhline(freq_clean.mean(), color='red', linestyle='--', label=f'Media: {freq_clean.mean():.0f}')
plt.legend()
plt.tight_layout()
plt.savefig('plots/desbalanceo_clases.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 4.5 TF-IDF Vectorización
X = df['text']
y = df['Category']

vectorizer = TfidfVectorizer(
    max_features=5000,       # Aumentado de 1000 a 5000 para mayor representación
    stop_words='english',
    ngram_range=(1, 2),      # Unigramas y bigramas
    sublinear_tf=True        # Suavizado logarítmico de frecuencias
)

X_tfidf = vectorizer.fit_transform(X)
print(f'Matriz TF-IDF shape: {X_tfidf.shape}')
print(f'Sparsity: {1 - X_tfidf.nnz / (X_tfidf.shape[0] * X_tfidf.shape[1]):.4f}')

In [ ]:
# 4.6 División Train / Validation / Test (70% / 15% / 15%)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X_tfidf, y, test_size=0.15, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.176, random_state=42, stratify=y_train_val
    # 0.176 de 85% ≈ 15% del total
)

print(f'Train:      {X_train.shape[0]} registros ({X_train.shape[0]/X_tfidf.shape[0]*100:.1f}%)')
print(f'Validation: {X_val.shape[0]} registros ({X_val.shape[0]/X_tfidf.shape[0]*100:.1f}%)')
print(f'Test:       {X_test.shape[0]} registros ({X_test.shape[0]/X_tfidf.shape[0]*100:.1f}%)')

---
## 5. Técnicas de Balanceo de Clases (AE 2.4)

Se comparan **3 estrategias de balanceo** para determinar cuál mejora más el rendimiento del modelo base (LinearSVC):

1. **RandomOverSampler**: duplica muestras de clases minoritarias aleatoriamente.
2. **RandomUnderSampler**: reduce muestras de clases mayoritarias.
3. **SMOTETomek**: oversampling sintético (SMOTE) + limpieza de frontera (Tomek Links).

In [ ]:
balanceo_resultados = {}

estrategias = {
    'Sin balanceo': (X_train, y_train),
}

# Oversampling
ros = RandomOverSampler(random_state=42)
X_ros, y_ros = ros.fit_resample(X_train, y_train)
estrategias['RandomOverSampler'] = (X_ros, y_ros)

# Undersampling
rus = RandomUnderSampler(random_state=42)
X_rus, y_rus = rus.fit_resample(X_train, y_train)
estrategias['RandomUnderSampler'] = (X_rus, y_rus)

# SMOTETomek
smotetomek = SMOTETomek(random_state=42)
X_smt, y_smt = smotetomek.fit_resample(X_train, y_train)
estrategias['SMOTETomek'] = (X_smt, y_smt)

# Comparar con LinearSVC (modelo rápido)
print('Comparando estrategias de balanceo con LinearSVC...\n')
for nombre, (Xb, yb) in estrategias.items():
    clf = LinearSVC(class_weight='balanced', max_iter=2000)
    clf.fit(Xb, yb)
    y_pred = clf.predict(X_val)
    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred, average='weighted', zero_division=0)
    balanceo_resultados[nombre] = {'Accuracy': acc, 'F1-weighted': f1, 'Tamaño Train': len(yb)}
    print(f'  {nombre}: Accuracy={acc:.4f} | F1={f1:.4f} | Train size={len(yb)}')

df_balanceo = pd.DataFrame(balanceo_resultados).T
print('\n=== Resumen Estrategias de Balanceo ===')
print(df_balanceo)

In [ ]:
# Visualización comparativa de estrategias de balanceo
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

metricas = ['Accuracy', 'F1-weighted']
colores = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

for i, metrica in enumerate(metricas):
    vals = [balanceo_resultados[k][metrica] for k in balanceo_resultados]
    nombres = list(balanceo_resultados.keys())
    bars = axes[i].bar(nombres, vals, color=colores, edgecolor='white', width=0.5)
    axes[i].set_title(f'{metrica} por Estrategia de Balanceo', fontweight='bold')
    axes[i].set_ylim(0, 1)
    axes[i].set_ylabel(metrica)
    axes[i].tick_params(axis='x', rotation=20)
    for bar, val in zip(bars, vals):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                     f'{val:.3f}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('plots/comparacion_balanceo.png', dpi=150, bbox_inches='tight')
plt.show()

mejor_balanceo = max(balanceo_resultados, key=lambda k: balanceo_resultados[k]['F1-weighted'])
print(f'\n✅ Mejor estrategia de balanceo: {mejor_balanceo}')

In [ ]:
# Seleccionar datos balanceados para el resto del entrenamiento
if mejor_balanceo == 'RandomOverSampler':
    X_bal, y_bal = X_ros, y_ros
elif mejor_balanceo == 'RandomUnderSampler':
    X_bal, y_bal = X_rus, y_rus
elif mejor_balanceo == 'SMOTETomek':
    X_bal, y_bal = X_smt, y_smt
else:
    X_bal, y_bal = X_train, y_train

print(f'Usando: {mejor_balanceo} — {len(y_bal)} muestras de entrenamiento')

---
## 6. Comparación de Técnicas ML Supervisadas (AE 2.2 / 2.3 / 2.5)

### Justificación de técnicas seleccionadas:

| Modelo | Justificación |
|--------|---------------|
| **LinearSVC** | Eficiente para espacios de alta dimensión (TF-IDF). Buena generalización en texto (Li et al., 2022). |
| **Logistic Regression** | Baseline sólido para clasificación multiclase. Rápido y explicable. |
| **Random Forest** | Robusto ante ruido y datos desbalanceados (Sahoo et al., 2022). Maneja no linealidades. |
| **Gradient Boosting** | Estado del arte en tabular/texto estructurado. Alta precisión con ajuste iterativo. |
| **Naive Bayes** | Altamente eficiente en clasificación de texto, especialmente con pocos datos. |

In [ ]:
from sklearn.naive_bayes import ComplementNB

modelos_ml = {
    'LinearSVC': LinearSVC(class_weight='balanced', max_iter=2000),
    'LogisticRegression': LogisticRegression(max_iter=1000, class_weight='balanced', C=1.0),
    'RandomForest': RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1),
    'ComplementNB': ComplementNB(),
}

results_ml = {}

print('Entrenando modelos ML...\n')
for nombre, modelo in modelos_ml.items():
    print(f'  ▶ {nombre}...')
    modelo.fit(X_bal, y_bal)

    # Validación
    y_pred_val = modelo.predict(X_val)
    # Test
    y_pred_test = modelo.predict(X_test)

    results_ml[nombre] = {
        'Acc_Val':  accuracy_score(y_val, y_pred_val),
        'F1_Val':   f1_score(y_val, y_pred_val, average='weighted', zero_division=0),
        'Rec_Val':  recall_score(y_val, y_pred_val, average='weighted', zero_division=0),
        'Prec_Val': precision_score(y_val, y_pred_val, average='weighted', zero_division=0),
        'Acc_Test': accuracy_score(y_test, y_pred_test),
        'F1_Test':  f1_score(y_test, y_pred_test, average='weighted', zero_division=0),
        'Rec_Test': recall_score(y_test, y_pred_test, average='weighted', zero_division=0),
        'Prec_Test':precision_score(y_test, y_pred_test, average='weighted', zero_division=0),
    }

    print(f'    Val  → Acc: {results_ml[nombre]["Acc_Val"]:.4f} | F1: {results_ml[nombre]["F1_Val"]:.4f}')
    print(f'    Test → Acc: {results_ml[nombre]["Acc_Test"]:.4f} | F1: {results_ml[nombre]["F1_Test"]:.4f}\n')

df_ml = pd.DataFrame(results_ml).T
print('\n=== RESUMEN MODELOS ML ===')
print(df_ml.round(4))

In [ ]:
# Visualización comparativa ML
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metricas_plot = [('Acc_Test', 'Accuracy (Test)'), ('F1_Test', 'F1-Score (Test)')]
palette = sns.color_palette('Set2', len(modelos_ml))

for ax, (col, titulo) in zip(axes, metricas_plot):
    vals = df_ml[col].sort_values(ascending=False)
    bars = ax.bar(vals.index, vals.values, color=palette, edgecolor='white', width=0.5)
    ax.set_title(titulo, fontweight='bold', fontsize=12)
    ax.set_ylim(0, 1)
    ax.set_ylabel('Score')
    ax.tick_params(axis='x', rotation=15)
    for bar, val in zip(bars, vals.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', fontsize=10, fontweight='bold')

plt.suptitle('Comparación de Modelos ML — Conjunto de Test', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('plots/comparacion_ml.png', dpi=150, bbox_inches='tight')
plt.show()

mejor_ml = df_ml['F1_Test'].idxmax()
print(f'\n✅ Mejor modelo ML: {mejor_ml} (F1-Test: {df_ml.loc[mejor_ml, "F1_Test"]:.4f})')

In [ ]:
# Matriz de confusión del mejor modelo ML
best_ml_model = modelos_ml[mejor_ml]
y_pred_best_ml = best_ml_model.predict(X_test)
cm_ml = confusion_matrix(y_test, y_pred_best_ml)

plt.figure(figsize=(14, 10))
sns.heatmap(cm_ml, cmap='Blues', xticklabels=False, yticklabels=False)
plt.title(f'Matriz de Confusión — {mejor_ml} (Test)', fontweight='bold', fontsize=13)
plt.xlabel('Predicho')
plt.ylabel('Real')
plt.tight_layout()
plt.savefig('plots/confusion_matrix_ml.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n=== Classification Report — {mejor_ml} ===')
print(classification_report(y_test, y_pred_best_ml, zero_division=0))

---
## 7. Aprendizaje No Supervisado — KMeans 

In [ ]:
# Reducción dimensional con TruncatedSVD (LSA)
svd = TruncatedSVD(n_components=50, random_state=42)
X_svd = svd.fit_transform(X_tfidf)
print(f'Varianza explicada (50 componentes): {svd.explained_variance_ratio_.sum():.4f}')

# KMeans con k = número de categorías
n_clusters = df['Category'].nunique()
print(f'\nEntrenando KMeans con k={n_clusters}...')
km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10, max_iter=300)
km_labels = km.fit_predict(X_svd)

# Silhouette score (muestra para eficiencia)
idx_sample = np.random.choice(len(X_svd), size=2000, replace=False)
sil_score = silhouette_score(X_svd[idx_sample], km_labels[idx_sample])
print(f'Silhouette Score (muestra 2000): {sil_score:.4f}')

# Comparación: clusters vs categorías reales
df_cluster = pd.DataFrame({'categoria': y.values, 'cluster': km_labels})
print(f'\nClusters generados: {len(np.unique(km_labels))}')
print(f'Categorías reales: {df["Category"].nunique()}')

In [ ]:
# Visualización reducción 2D de clusters
svd_2d = TruncatedSVD(n_components=2, random_state=42)
X_2d = svd_2d.fit_transform(X_tfidf)

idx_vis = np.random.choice(len(X_2d), size=3000, replace=False)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(X_2d[idx_vis, 0], X_2d[idx_vis, 1],
            c=km_labels[idx_vis], cmap='tab20', alpha=0.4, s=10)
plt.title('KMeans — Clusters (SVD 2D)', fontweight='bold')
plt.xlabel('Componente 1')
plt.ylabel('Componente 2')

le_vis = LabelEncoder()
y_enc_vis = le_vis.fit_transform(y.values)

plt.subplot(1, 2, 2)
plt.scatter(X_2d[idx_vis, 0], X_2d[idx_vis, 1],
            c=y_enc_vis[idx_vis], cmap='tab20', alpha=0.4, s=10)
plt.title('Categorías Reales (SVD 2D)', fontweight='bold')
plt.xlabel('Componente 1')
plt.ylabel('Componente 2')

plt.tight_layout()
plt.savefig('plots/kmeans_vs_real.png', dpi=150, bbox_inches='tight')
plt.show()

print('Análisis: Un Silhouette Score bajo indica alta superposición entre categorías,')
print('lo que justifica el uso de modelos supervisados para esta tarea.')

---
## 8. Deep Learning — 3 Arquitecturas 


In [ ]:
# Preparar datos para DL
le = LabelEncoder()
y_train_enc = le.fit_transform(y_bal)
y_val_enc   = le.transform(y_val)
y_test_enc  = le.transform(y_test)

n_classes = len(le.classes_)

y_train_cat = to_categorical(y_train_enc, num_classes=n_classes)
y_val_cat   = to_categorical(y_val_enc,   num_classes=n_classes)
y_test_cat  = to_categorical(y_test_enc,  num_classes=n_classes)

X_train_dl = X_bal.toarray().astype('float32')
X_val_dl   = X_val.toarray().astype('float32')
X_test_dl  = X_test.toarray().astype('float32')

input_dim = X_train_dl.shape[1]
print(f'Clases: {n_classes} | Input dim: {input_dim}')
print(f'Train DL: {X_train_dl.shape} | Val: {X_val_dl.shape} | Test: {X_test_dl.shape}')

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
]

In [ ]:
# ARQUITECTURA 1: MLP Simple

tf.random.set_seed(42)

model_mlp = Sequential([
    Dense(256, activation='relu', input_shape=(input_dim,)),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(n_classes, activation='softmax')
], name='MLP_Simple')

model_mlp.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_mlp.summary()

history_mlp = model_mlp.fit(
    X_train_dl, y_train_cat,
    epochs=30,
    batch_size=64,
    validation_data=(X_val_dl, y_val_cat),
    callbacks=callbacks,
    verbose=1
)

loss_mlp, acc_mlp = model_mlp.evaluate(X_test_dl, y_test_cat, verbose=0)
y_pred_mlp = np.argmax(model_mlp.predict(X_test_dl, verbose=0), axis=1)
f1_mlp = f1_score(y_test_enc, y_pred_mlp, average='weighted', zero_division=0)
print(f'\n✅ MLP Simple — Test Accuracy: {acc_mlp:.4f} | F1: {f1_mlp:.4f}')

In [ ]:
# ARQUITECTURA 2: MLP Profundo con BatchNorm

tf.random.set_seed(42)

model_mlp_deep = Sequential([
    Dense(512, activation='relu', input_shape=(input_dim,)),
    BatchNormalization(),
    Dropout(0.4),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.2),
    Dense(n_classes, activation='softmax')
], name='MLP_Profundo_BatchNorm')

model_mlp_deep.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_mlp_deep.summary()

history_mlp_deep = model_mlp_deep.fit(
    X_train_dl, y_train_cat,
    epochs=30,
    batch_size=64,
    validation_data=(X_val_dl, y_val_cat),
    callbacks=callbacks,
    verbose=1
)

loss_deep, acc_deep = model_mlp_deep.evaluate(X_test_dl, y_test_cat, verbose=0)
y_pred_deep = np.argmax(model_mlp_deep.predict(X_test_dl, verbose=0), axis=1)
f1_deep = f1_score(y_test_enc, y_pred_deep, average='weighted', zero_division=0)
print(f'\n✅ MLP Profundo — Test Accuracy: {acc_deep:.4f} | F1: {f1_deep:.4f}')

In [ ]:
# ARQUITECTURA 3: CNN 1D
tf.random.set_seed(42)

model_cnn = Sequential([
    Reshape((input_dim, 1), input_shape=(input_dim,)),
    Conv1D(128, kernel_size=5, activation='relu', padding='same'),
    BatchNormalization(),
    Dropout(0.3),
    Conv1D(64, kernel_size=3, activation='relu', padding='same'),
    GlobalMaxPooling1D(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(n_classes, activation='softmax')
], name='CNN_1D')

model_cnn.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_cnn.summary()

history_cnn = model_cnn.fit(
    X_train_dl, y_train_cat,
    epochs=30,
    batch_size=64,
    validation_data=(X_val_dl, y_val_cat),
    callbacks=callbacks,
    verbose=1
)

loss_cnn, acc_cnn = model_cnn.evaluate(X_test_dl, y_test_cat, verbose=0)
y_pred_cnn = np.argmax(model_cnn.predict(X_test_dl, verbose=0), axis=1)
f1_cnn = f1_score(y_test_enc, y_pred_cnn, average='weighted', zero_division=0)
print(f'\n✅ CNN 1D — Test Accuracy: {acc_cnn:.4f} | F1: {f1_cnn:.4f}')

In [ ]:
# Curvas de convergencia — las 3 arquitecturas DL
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

historiales = [
    (history_mlp,      'MLP Simple'),
    (history_mlp_deep, 'MLP Profundo'),
    (history_cnn,      'CNN 1D')
]

for col, (hist, nombre) in enumerate(historiales):
    # Accuracy
    axes[0, col].plot(hist.history['accuracy'],     label='Train', color='steelblue')
    axes[0, col].plot(hist.history['val_accuracy'], label='Val',   color='orange', linestyle='--')
    axes[0, col].set_title(f'{nombre} — Accuracy', fontweight='bold')
    axes[0, col].set_xlabel('Época')
    axes[0, col].set_ylabel('Accuracy')
    axes[0, col].legend()
    axes[0, col].grid(alpha=0.3)

    # Loss
    axes[1, col].plot(hist.history['loss'],     label='Train', color='steelblue')
    axes[1, col].plot(hist.history['val_loss'], label='Val',   color='orange', linestyle='--')
    axes[1, col].set_title(f'{nombre} — Loss', fontweight='bold')
    axes[1, col].set_xlabel('Época')
    axes[1, col].set_ylabel('Loss')
    axes[1, col].legend()
    axes[1, col].grid(alpha=0.3)

plt.suptitle('Curvas de Convergencia — Arquitecturas Deep Learning', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('plots/curvas_convergencia_dl.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Resumen Comparativo General (AE 2.5)

In [ ]:
# Tabla resumen: ML + DL
resultados_dl = {
    'MLP Simple':     {'Acc_Test': acc_mlp,  'F1_Test': f1_mlp,  'Tipo': 'DL'},
    'MLP Profundo':   {'Acc_Test': acc_deep, 'F1_Test': f1_deep, 'Tipo': 'DL'},
    'CNN 1D':         {'Acc_Test': acc_cnn,  'F1_Test': f1_cnn,  'Tipo': 'DL'},
}

# Combinar ML y DL
resumen_completo = {}
for nombre, vals in results_ml.items():
    resumen_completo[nombre] = {
        'Accuracy': vals['Acc_Test'],
        'F1-Score': vals['F1_Test'],
        'Precision': vals['Prec_Test'],
        'Recall': vals['Rec_Test'],
        'Tipo': 'ML'
    }
for nombre, vals in resultados_dl.items():
    resumen_completo[nombre] = {
        'Accuracy': vals['Acc_Test'],
        'F1-Score': vals['F1_Test'],
        'Precision': None,
        'Recall': None,
        'Tipo': vals['Tipo']
    }

df_resumen = pd.DataFrame(resumen_completo).T
df_resumen = df_resumen.sort_values('F1-Score', ascending=False)
print('=== TABLA COMPARATIVA FINAL (Test Set) ===')
print(df_resumen.round(4).to_string())

In [ ]:
# Gráfico comparativo final ML vs DL
modelos_nombres = list(df_resumen.index)
f1_vals  = df_resumen['F1-Score'].astype(float).values
acc_vals = df_resumen['Accuracy'].astype(float).values
tipos    = df_resumen['Tipo'].values

colores_barra = ['#4C72B0' if t == 'ML' else '#DD8452' for t in tipos]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (vals, titulo) in zip(axes, [(f1_vals, 'F1-Score (Test)'), (acc_vals, 'Accuracy (Test)')]):
    bars = ax.bar(modelos_nombres, vals, color=colores_barra, edgecolor='white', width=0.6)
    ax.set_title(titulo, fontweight='bold', fontsize=12)
    ax.set_ylim(0, 1)
    ax.set_ylabel('Score')
    ax.tick_params(axis='x', rotation=20)
    for bar, val in zip(bars, vals):
        if not np.isnan(val):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{val:.3f}', ha='center', fontsize=9, fontweight='bold')

# Leyenda manual
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#4C72B0', label='ML'), Patch(facecolor='#DD8452', label='Deep Learning')]
axes[0].legend(handles=legend_elements, loc='lower right')

plt.suptitle('Comparación Final — ML vs Deep Learning', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('plots/comparacion_final_ml_dl.png', dpi=150, bbox_inches='tight')
plt.show()

mejor_global = df_resumen['F1-Score'].astype(float).idxmax()
print(f'\n🏆 MEJOR MODELO GLOBAL: {mejor_global}')
print(f'   F1-Score: {df_resumen.loc[mejor_global, "F1-Score"]:.4f}')
print(f'   Accuracy: {df_resumen.loc[mejor_global, "Accuracy"]:.4f}')

---
## 10. Explicabilidad con SHAP 

In [ ]:
# SHAP sobre el mejor modelo ML (LinearSVC o LR)
# Usar el mejor modelo ML para SHAP (más compatible con shap.LinearExplainer)
shap_model = modelos_ml.get('LinearSVC') or modelos_ml.get('LogisticRegression')
shap_model_name = 'LinearSVC' if 'LinearSVC' in modelos_ml else 'LogisticRegression'

print(f'Calculando SHAP para: {shap_model_name}...')

# Muestra reducida para eficiencia
idx_shap = np.random.choice(X_test.shape[0], size=min(200, X_test.shape[0]), replace=False)
X_shap = X_test[idx_shap]

explainer = shap.LinearExplainer(shap_model, X_bal, feature_perturbation='interventional')
shap_values = explainer.shap_values(X_shap)

feature_names = vectorizer.get_feature_names_out()

# Summary plot — importancia global de features
plt.figure(figsize=(12, 6))
shap.summary_plot(
    shap_values if isinstance(shap_values, np.ndarray) else np.array(shap_values).mean(axis=0),
    X_shap,
    feature_names=feature_names,
    max_display=20,
    plot_type='bar',
    show=False
)
plt.title(f'SHAP — Top 20 Features más Importantes ({shap_model_name})', fontweight='bold')
plt.tight_layout()
plt.savefig('plots/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ SHAP calculado correctamente')

---
## 11. Mejoras Propuestas (AE 2.6)

### Limitaciones detectadas:
- El desbalanceo de clases (569 vs 2 registros) afecta el recall en clases minoritarias.
- El modelo TF-IDF pierde información semántica y contextual del texto.
- El MLP no captura dependencias secuenciales en el texto.

### Mejoras propuestas:

**Mejora 1: Embeddings preentrenados (BERT / sentence-transformers)**  
Reemplazar TF-IDF por embeddings contextuales mejora la representación semántica, especialmente para descripciones cortas. Ferrag et al. (2021) reportaron mejoras del 8–12% con BERT en clasificación de ataques.

**Mejora 2: Ensemble Stacking**  
Combinar las predicciones de LinearSVC + LogisticRegression + MLP mediante un meta-clasificador (Stacking) puede reducir la varianza y mejorar el F1 en clases difíciles.

**Mejora 3: Ajuste de umbral de decisión por clase**  
Optimizar el umbral de clasificación individualmente por categoría, especialmente para las clases minoritarias, puede mejorar el recall sin sacrificar precisión.

In [ ]:
# Mejora 2: Ensemble Voting (Soft/Hard) como propuesta implementable
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV

print('Entrenando ensemble (Voting) como mejora propuesta...\n')

# Calibrar LinearSVC para obtener probabilidades
svc_cal = CalibratedClassifierCV(
    LinearSVC(class_weight='balanced', max_iter=2000), cv=3
)
svc_cal.fit(X_bal, y_bal)

lr_model = LogisticRegression(max_iter=1000, class_weight='balanced', C=1.0)
lr_model.fit(X_bal, y_bal)

# Promedio de probabilidades (soft voting)
proba_svc = svc_cal.predict_proba(X_test)
proba_lr  = lr_model.predict_proba(X_test)
proba_ensemble = (proba_svc + proba_lr) / 2

# Mapear clases al labelencoder del modelo
classes_lr = lr_model.classes_
y_pred_ensemble_idx = np.argmax(proba_ensemble, axis=1)
y_pred_ensemble = classes_lr[y_pred_ensemble_idx]

acc_ens = accuracy_score(y_test, y_pred_ensemble)
f1_ens  = f1_score(y_test, y_pred_ensemble, average='weighted', zero_division=0)
rec_ens = recall_score(y_test, y_pred_ensemble, average='weighted', zero_division=0)

print(f'Ensemble (SVC + LR Soft Voting):')
print(f'  Accuracy: {acc_ens:.4f}')
print(f'  F1-Score: {f1_ens:.4f}')
print(f'  Recall:   {rec_ens:.4f}')
print(f'\nComparación vs mejor modelo individual ({mejor_ml}):')
print(f'  ΔF1 = {f1_ens - results_ml[mejor_ml]["F1_Test"]:+.4f}')

---
## 12. Exportación del Modelo Final

In [ ]:
import os
os.makedirs('model', exist_ok=True)

# Seleccionar el mejor modelo ML para producción
modelo_final = modelos_ml[mejor_ml]

# Re-entrenar con train+val para maximizar datos antes de producción
from scipy.sparse import vstack as sp_vstack
X_trainval = sp_vstack([X_bal, X_val])
import numpy as np
y_trainval = np.concatenate([y_bal, y_val])

modelo_final.fit(X_trainval, y_trainval)
y_pred_final = modelo_final.predict(X_test)

acc_final = accuracy_score(y_test, y_pred_final)
f1_final  = f1_score(y_test, y_pred_final, average='weighted', zero_division=0)

print(f'Modelo final ({mejor_ml}) entrenado en Train+Val:')
print(f'  Test Accuracy: {acc_final:.4f}')
print(f'  Test F1-Score: {f1_final:.4f}')

# Guardar modelo y vectorizador
joblib.dump(modelo_final, 'model/model.pkl')
joblib.dump(vectorizer,   'model/vectorizer.pkl')

print('\n✅ Archivos guardados:')
print('   model/model.pkl')
print('   model/vectorizer.pkl')

In [ ]:
# Resumen ejecutivo final
print('=' * 60)
print('         RESUMEN EJECUTIVO — PROYECTO FINAL')
print('=' * 60)
print(f'Dataset:          14.133 registros | 63 categorías')
print(f'Features:         TF-IDF (5000 features, bigramas)')
print(f'Mejor balanceo:   {mejor_balanceo}')
print(f'Mejor modelo ML:  {mejor_ml}')
print(f'  Accuracy Test:  {results_ml[mejor_ml]["Acc_Test"]:.4f}')
print(f'  F1 Test:        {results_ml[mejor_ml]["F1_Test"]:.4f}')
print(f'Mejor arq. DL:    Según curvas de convergencia')
print(f'Explicabilidad:   SHAP LinearExplainer')
print(f'Integración:      Flask + deep_translator (app.py)')
print('=' * 60)